# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate all available RecordSets and their `@id`, as well as the Fields and Columns they contain.

In [ ]:
record_sets = list(dataset.record_sets())  # returns a list of RecordSet metadata

if not record_sets:
    print("No RecordSets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs['name']}")
        print(f"  Description: {rs.get('description', 'No description.')}")
        
        # List fields
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType','')}")
            else:
                print(f"    Field ref: {field}")
        
        # List columns
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        print("  Columns:")
        for column in columns:
            if isinstance(column, dict):
                print(f"    Column @id: {column['@id']}, name: {column.get('name','')}, dataType: {column.get('dataType','')}")
            else:
                print(f"    Column ref: {column}")
        print()

### Example record examination

Load and print some sample records from the **first record set** (by `@id`). This will help in identifying the field names and data structure for subsequent steps.

In [ ]:
# Get the @id of the first record set (if present)
record_sets_list = list(dataset.record_sets())
if record_sets_list:
    first_record_set_id = record_sets_list[0]['@id']
    print(f"Sample records for record set: {first_record_set_id}\n")
    for idx, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if idx>=2:  # Show only first 3 records
            break
else:
    print("No record sets with records found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}

record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
print(f"Record sets for extraction: {record_set_ids}\n")

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    else:
        print(f"No records loaded for RecordSet @id: {record_set_id}")

# Print the columns of the first dataframe for examination
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in main DataFrame for {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In the following example, we'll identify a numeric field (e.g. a protein abundance or molecular weight column) by `@id` and perform EDA. Make sure to substitute these ids based on your output from the previous sections.

In [ ]:
# ----- Change this to match the actual numeric field id in your dataset -----
# For demonstration, suppose the column 'Molecular Weight' (MW) exists and is called 'MW' with field @id 'MW'
main_rs_id = list(dataframes.keys())[0] if dataframes else None
if main_rs_id:
    df = dataframes[main_rs_id]
    print(f"\nAvailable columns: {df.columns.tolist()}")
    
    # Let's pick a likely numeric column. Update as needed.
    numeric_candidates = [c for c in df.columns if df[c].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use first found numeric column
    else:
        print("No numeric columns detected.\n")
        numeric_field = None

    if numeric_field:
        # Drop missing values for this demo
        df = df.dropna(subset=[numeric_field])
        # Optionally, cast to float
        df[numeric_field] = df[numeric_field].astype(float)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a possible categorical column
        categorical_candidates = [c for c in df.columns if df[c].dtype==object and c!=numeric_field]
        if categorical_candidates:
            group_field = categorical_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a categorical group_field exists, plot boxplots
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded using `mlcroissant` according to its Croissant schema.
- All processing used schema `@id` identifiers for reliable mapping of record sets and fields.
- Example EDA and visualization steps were demonstrated for numeric and categorical fields. To go deeper, use domain knowledge to choose specific fields and perform custom analysis.

For further work, reference the output of the Data Overview section to use specific `@id` values for fields of interest.